# Phase 1 Final Remediation Plan

This notebook is orchestration only. It records a claim-closed remediation plan after the reviewed Phase 1 final claim-state closeout.

Scientific integrity rules:

- This notebook does not rerun models, controls, calibration, influence, or reporting.
- It does not relax thresholds, edit logits, drop subjects, or reinterpret failed governance surfaces as pass.
- It records allowed remediation classes and revision-policy guardrails only.
- The source closeout must already show `claims_opened = false` and `final_claim_blocked = true`.
- Any future remediation remains claim-closed until the affected artifacts are revised, rerun, reconciled, and reviewed.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed closeout source and keep launch disabled by default.

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Pin the reviewed closeout source. Replace only after reviewing a newer closeout run.
FINAL_CLAIM_STATE_CLOSEOUT_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_claim_state_closeout/20260422T083855265838Z')

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_remediation_plan'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_remediation_plan_manual_hold'

RUN_FINAL_REMEDIATION_PLAN = False
REQUIRED_ACK = 'I reviewed final remediation planning and understand it keeps Phase 1 claims closed while blockers remain'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_remediation_plan_v1_20260422'

print('Notebook fix marker:', FIX_MARKER)
print('Closeout source:', FINAL_CLAIM_STATE_CLOSEOUT_RUN)
print('Run flag:', RUN_FINAL_REMEDIATION_PLAN)


In [ ]:
# Cell 3 - Utilities and prereg/closeout validation.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '
', encoding='utf-8')

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_prereg_bundle(path: Path) -> Path:
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Found prereg bundles:', len(candidates))
    for candidate in candidates:
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_IDENTITY_HASH:
            return candidate
    raise FileNotFoundError(f'No locked prereg bundle found for expected identity hash under {ARTIFACT_ROOT / "prereg"}')

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
FINAL_CLAIM_STATE_CLOSEOUT_RUN = Path(FINAL_CLAIM_STATE_CLOSEOUT_RUN)

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

closeout_summary = read_json(FINAL_CLAIM_STATE_CLOSEOUT_RUN / 'phase1_final_claim_state_closeout_summary.json')
closeout_disposition = read_json(FINAL_CLAIM_STATE_CLOSEOUT_RUN / 'phase1_final_claim_disposition.json')
closeout_blocker_table = read_json(FINAL_CLAIM_STATE_CLOSEOUT_RUN / 'phase1_final_blocker_table.json')

assert closeout_summary.get('status') == 'phase1_final_claim_blocked_fail_closed', closeout_summary.get('status')
assert closeout_summary.get('claims_opened') is False, 'Closeout must not have opened claims.'
assert closeout_summary.get('final_claim_blocked') is True, 'Closeout must be fail-closed before remediation planning.'
assert closeout_summary.get('revision_required_for_remediation') is True, 'Remediation must require revision policy review.'
assert closeout_disposition.get('claims_opened') is False, 'Disposition must preserve closed claims.'
assert set(closeout_summary.get('blocking_surfaces', [])) >= {'controls', 'calibration', 'influence'}

preflight = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_remediation_plan', '--help'],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)
runner_available = preflight.returncode == 0
print('Runner available:', runner_available)
assert runner_available, preflight.stderr


In [ ]:
# Cell 4 - Record manual-hold plan and exact CLI command.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / created_utc
cmd = [
    'python', '-m', 'src.cli', 'phase1_final_remediation_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--claim-state-closeout-run', str(FINAL_CLAIM_STATE_CLOSEOUT_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--remediation-config', 'configs/phase1/final_remediation_plan.json',
]
plan = {
    'status': 'phase1_final_remediation_plan_manual_hold_recorded',
    'created_utc': created_utc,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_REMEDIATION_PLAN,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'final_claim_state_closeout_run': str(FINAL_CLAIM_STATE_CLOSEOUT_RUN),
    'closeout_status': closeout_summary.get('status'),
    'closeout_claims_opened': closeout_summary.get('claims_opened'),
    'closeout_blocking_surfaces': closeout_summary.get('blocking_surfaces', []),
    'closeout_claim_blockers': closeout_summary.get('claim_blockers', []),
    'command': cmd,
    'scientific_limit': 'This plan records claim-closed remediation scope only; it does not rerun analyses or open claims.',
}
write_json(plan_dir / 'phase1_final_remediation_plan_manual_hold.json', plan)
print('Plan source of truth:', plan_dir)
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual hold or launch.

if not RUN_FINAL_REMEDIATION_PLAN:
    hold = {
        'status': 'phase1_final_remediation_plan_manual_hold',
        'plan_dir': str(plan_dir),
        'run_flag': RUN_FINAL_REMEDIATION_PLAN,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    }
    print(json.dumps(hold, indent=2))
    print('HELD: Remediation plan was not launched because manual flag is False.')
    print('NEXT: review the closeout and plan artifact, then rerun only with explicit claim-closed acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch remediation planning without explicit claim-closed acknowledgement.')
else:
    write_json(plan_dir / 'phase1_final_remediation_plan_launch_review.json', {
        'status': 'phase1_final_remediation_plan_launch_reviewed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'ack': MANUAL_LAUNCH_ACK,
        'claims_opened': False,
        'scientific_limit': 'Launch records a plan only; it must not alter failed governance evidence.',
    })
    run(cmd, cwd=REPO_DIR)
    print('Final remediation plan command completed. Review generated artifacts before any future remediation work.')


In [ ]:
# Cell 6 - Inspect latest remediation-plan output.

latest_txt = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest_txt.exists())
if latest_txt.exists():
    latest_run = Path(latest_txt.read_text(encoding='utf-8').strip())
else:
    runs = sorted([path for path in OUTPUT_ROOT.iterdir() if path.is_dir()]) if OUTPUT_ROOT.exists() else []
    latest_run = runs[-1] if runs else None

if latest_run is None:
    print('No remediation-plan output yet; this is expected for manual hold.')
else:
    print('Latest final remediation-plan output:', latest_run)
    expected_files = [
        'phase1_final_remediation_plan_inputs.json',
        'phase1_final_remediation_plan_summary.json',
        'phase1_final_remediation_plan_report.md',
        'phase1_final_remediation_source_links.json',
        'phase1_final_remediation_input_validation.json',
        'phase1_final_remediation_blocker_review.json',
        'phase1_final_remediation_workplan.json',
        'phase1_final_remediation_guardrails.json',
        'phase1_final_remediation_revision_checklist.json',
        'phase1_final_remediation_claim_state.json',
        'phase1_final_remediation_decision_memo.md',
    ]
    for name in expected_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_remediation_plan_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'claim_ready': summary.get('claim_ready'),
        'claims_opened': summary.get('claims_opened'),
        'final_claim_blocked': summary.get('final_claim_blocked'),
        'blocking_surfaces': summary.get('blocking_surfaces'),
        'next_step': summary.get('next_step'),
        'scientific_limit': summary.get('scientific_limit'),
    }, indent=2))


In [ ]:
# Cell 7 - Assertions and local review note.

if RUN_FINAL_REMEDIATION_PLAN:
    assert latest_run is not None, 'Expected remediation-plan output after launch.'
    summary = read_json(latest_run / 'phase1_final_remediation_plan_summary.json')
    blocker_review = read_json(latest_run / 'phase1_final_remediation_blocker_review.json')
    workplan = read_json(latest_run / 'phase1_final_remediation_workplan.json')
    guardrails = read_json(latest_run / 'phase1_final_remediation_guardrails.json')
    claim_state = read_json(latest_run / 'phase1_final_remediation_claim_state.json')

    assert summary.get('status') == 'phase1_final_remediation_plan_recorded', summary
    assert summary.get('claim_ready') is False
    assert summary.get('claims_opened') is False
    assert summary.get('final_claim_blocked') is True
    assert claim_state.get('headline_phase1_claim_open') is False
    assert set(blocker_review.get('blocking_surfaces', [])) >= {'controls', 'calibration', 'influence'}
    assert 'threshold_relaxation_after_observed_failure' in guardrails.get('not_allowed_remediation_classes', [])
    assert workplan.get('work_items'), 'Expected remediation work items for failed governance surfaces.'

    review_note = {
        'status': 'phase1_final_remediation_plan_review_pass_claim_closed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_run),
        'checks_passed': [
            'required_artifacts_present',
            'claim_ready_false',
            'claims_opened_false',
            'final_claim_blocked_true',
            'controls_calibration_influence_remain_blocking_surfaces',
            'threshold_relaxation_marked_not_allowed',
            'workplan_records_revision_scoped_remediation_only',
        ],
        'allowed_interpretation': 'Engineering remediation planning only.',
        'not_allowed_interpretation': 'Do not treat this plan as evidence that controls, calibration, or influence passed.',
        'not_ok_to_claim': guardrails.get('not_ok_to_claim', []),
    }
    write_json(latest_run / 'phase1_final_remediation_plan_review_note.json', review_note)
    print('Review note written:', latest_run / 'phase1_final_remediation_plan_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch flag is False. This is expected during first-pass manual hold.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Final Remediation Plan Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_REMEDIATION_PLAN)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Final claim-state closeout run:', FINAL_CLAIM_STATE_CLOSEOUT_RUN)

if not RUN_FINAL_REMEDIATION_PLAN:
    print('HELD: Remediation planning was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
else:
    print('Latest final remediation-plan output:', latest_run)
    print('Blocking surfaces:', summary.get('blocking_surfaces'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Next step:', summary.get('next_step'))
    print('CHECK REQUIRED: Review phase1_final_remediation_decision_memo.md before editing controls, calibration or influence code/config.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
